<a href="https://colab.research.google.com/github/elisacapilli/ai-notebooks/blob/upload-notebooks/10_stats_Gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --force-reinstall transformers vllm

In [ ]:
!pip install --upgrade numpy torch transformers vllm --no-cache-dir

In [ ]:
import numpy
import transformers
import vllm

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import numpy as np
import pandas as pd
import torch
import random
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoConfig, TrainingArguments, Trainer, AutoModelForCausalLM, pipeline
tqdm.pandas()
import os
os.environ["WANDB_MODE"] = "disabled"

from vllm import LLM, SamplingParams

# import outlines
from pydantic import BaseModel
from lmformatenforcer import RegexParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn
from lmformatenforcer.integrations.vllm import build_vllm_token_enforcer_tokenizer_data, build_vllm_logits_processor

CACHE = '/content/.cache/huggingface/'

from huggingface_hub import login
login(token="HF_TOKEN")

import warnings
warnings.filterwarnings('ignore')

In [ ]:
model_id = 'google/gemma-2-2b-it'

llm = LLM(
    model=model_id,
    tokenizer=model_id,
    tokenizer_mode="mistral" if "mistral" in model_id else "auto",
    trust_remote_code=True,
    dtype=torch.float16,
    max_model_len=2048*2,
    tensor_parallel_size=1,
    download_dir=CACHE,
)

tokenizer =  build_vllm_token_enforcer_tokenizer_data(llm)

In [ ]:
#import pandas as pd
from IPython.display import display

df = pd.read_excel("df_facct.xlsx")

In [ ]:
import pandas as pd

def select_random_rows(file_path):
    df = pd.read_excel(file_path)
    random_rows = df.sample(n=10)
    result = []
    for _, row in random_rows.iterrows():
        truth_value = row['fact_check_ground_truth_value']
        statement = row['statement_text']

        result.append({
            "Truth Value": truth_value,
            "Statement": statement
        })

    return result

file_path = "df_facct.xlsx"
random_rows_data = select_random_rows(file_path)

result_df = pd.DataFrame(random_rows_data)

display(result_df)


In [ ]:
import pandas as pd

def select_random_rows(file_path):
    df = pd.read_excel(file_path)
    random_rows = df.sample(n=10)

    result = []
    for _, row in random_rows.iterrows():
        truth_value = row['fact_check_ground_truth_value']
        statement = row['statement_text']

        result.append({
            "Truth Value": truth_value,
            "Statement": statement
        })

    return pd.DataFrame(result)


def generate_prompt_fstring(row):
    statement = row['Statement']
    prompt = f"""
    You are a professional fact-checking AI. Your task is to evaluate the truthfulness of each statement in the given set using real-world facts and verified evidence using the Politifact scale.
    PolitiFact Factuality Scale (integer values only):
    0 = Pants on Fire — statement completely false and absurd
    1 = False — statement entirely incorrect
    2 = Mostly False — statement contains significant falsehoods or misleading elements
    3 = Half True — statement mixes truth and falsehood
    4 = Mostly True — statement mostly accurate with minor errors
    5 = True — statement fully accurate and supported by verifiable facts

    Your task:
    - Evaluate the truthfulness of the statement based on factual, verified information.
    - Based on the evaluations, provide a single integer answer from 0 to 5, representing your factuality rating for the statement.
    - Provide no explanations or additional text.

    Statement: "{statement}"
    Answer:"""
    return prompt

def get_model_response(prompt):
    logits_processor = build_vllm_logits_processor(tokenizer, RegexParser(r"[0-5]"))

    sampling_params = SamplingParams(
        temperature=0.1,
        max_tokens=1,
        logits_processors=[logits_processor],
    )

    outputs = llm.generate([prompt], sampling_params, use_tqdm=True)
    numeric_response = outputs[0].outputs[0].text.strip()
    return numeric_response

file_path = "df_facct.xlsx"
result_df = select_random_rows(file_path)

results = []

for idx, row in result_df.iterrows():
    prompt = generate_prompt_fstring(row)
    model_response = get_model_response(prompt)

    results.append({
        "Original Statement": row["Statement"],
        "Ground Truth Value": row["Truth Value"],
        "Generated Prompt": prompt,
        "Model Response": model_response
    })

final_results_df = pd.DataFrame(results)

# 5. Visualizza i risultati
display(final_results_df)
